# **Preparar el Entorno**

In [16]:
%pip install sodapy pandas tqdm pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\jaena\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [23]:
%pip install fastparquet

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/728.0 kB ? eta -:--:--
   -------------- ------------------------- 262.1/728.0 kB ? eta -:--:--
   ---------------------------------------- 728.0/728.0 kB 4.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 14.7 MB/s  0:00:00

   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ---------------------------------------- 0/3 [fsspec]
   ------------- -------------------------- 1/3 [cramjam]
   -------------------------- ------------- 2/3 [fastparquet]
   ---------------------------------------- 3/3 [fastparquet]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\jaena\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [17]:
# Se importan las librerías
import pandas as pd
from sodapy import Socrata
from tqdm import tqdm

# **Esquema del CSV: Diccionario de Datos**

In [18]:
# 1. Configuración de Socrata
DOMAIN = "www.datos.gov.co"
DATASET_ID = "kgxf-xxbe"  # ID del conjunto de datos "Resultados únicos Saber 11"
APP_TOKEN = "9GeA2HqSj6xVxhcKJnuBF5VnU" # Token de la aplicación (es del rolo)

### Data Frame

#### descarga del data set

In [15]:
client = Socrata(DOMAIN, APP_TOKEN, timeout=120)

# 2. Consultar el total de registros en la base de datos
print("Consultando la cantidad total de registros...")
count_res = client.get(DATASET_ID, select="count(*)")
total_registros = int(count_res[0]['count'])
print(f"Total de registros a descargar: {total_registros:,}")

# 3. Parámetros de paginación
LIMIT = 50000  # Máximo de filas por petición permitido por Socrata
OFFSET = 0
todas_las_filas = []

# 4. Descarga paginada con barra de progreso
pbar = tqdm(total=total_registros, desc="Descargando datos")

while OFFSET < total_registros:
    # Petición por bloque
    batch = client.get(DATASET_ID, limit=LIMIT, offset=OFFSET)
    
    if not batch:
        break
        
    todas_las_filas.extend(batch)
    OFFSET += LIMIT
    pbar.update(len(batch))

pbar.close()

# 5. Convertir a DataFrame de Pandas
print("Creando el DataFrame...")
df = pd.DataFrame.from_records(todas_las_filas)

print(f"¡Proceso completado! DataFrame creado con {df.shape[0]:,} filas y {df.shape[1]} columnas.")

Consultando la cantidad total de registros...
Total de registros a descargar: 7,109,704


Descargando datos: 100%|██████████| 7109704/7109704 [23:11<00:00, 5108.95it/s]


Creando el DataFrame...
¡Proceso completado! DataFrame creado con 7,109,704 filas y 51 columnas.


In [24]:
df.to_parquet("resultados_saber_11.parquet", engine="fastparquet", index=False)

#### carga del data frame 


In [26]:
import pyarrow.parquet as pq

# Lee la tabla con pyarrow y la convierte a pandas de forma directa
table = pq.read_table("resultados_saber_11.parquet")
df = table.to_pandas()

print(f"Cargado con éxito: {df.shape[0]:,} filas y {df.shape[1]} columnas.")

Cargado con éxito: 7,109,704 filas y 51 columnas.


In [27]:
df

,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,...,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20131,CR,SB11201310000414,URBANO,N,B,ACADÉMICO,311848000812,311848000812,11,...,Si,Si,Si,B+,94,88,NaN,NaN,NaN,NaN
1,20194,TI,SB11201940464873,RURAL,N,A,TÉCNICO/ACADÉMICO,141016000305,241016000342,41,...,Si,Si,Si,B1,71,66,70,65,69,339
2,20194,TI,SB11201940464873,RURAL,N,A,TÉCNICO/ACADÉMICO,141016000305,241016000342,41,...,Si,Si,Si,B1,71,66,70,65,69,339
3,20122,TI,SB11201220204399,URBANO,N,A,TÉCNICO/ACADÉMICO,163130000718,163130000718,63,...,Si,Si,No,A1,48,45,NaN,NaN,NaN,NaN
4,20132,TI,SB11201320464198,URBANO,N,A,TÉCNICO,119001000559,119001000559,19,...,Si,Si,Si,A-,43,52,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7109699,20221,TI,AC202210003698,URBANO,S,B,ACADÉMICO,311001107513,311001107513,11,...,Si,Si,Si,B1,76,55,54,56,56,284
7109700,20221,CC,AC202210034448,URBANO,N,A,ACADÉMICO,311001105928,311001105928,11,...,Si,Si,Si,A1,48,53,38,45,50,233
7109701,20221,TI,AC202210006814,URBANO,S,B,TÉCNICO/ACADÉMICO,311001019568,311001019568,11,...,Si,Si,Si,B1,73,67,57,62,63,315
7109702,20221,CC,AC202210029659,URBANO,N,A,ACADÉMICO,311001091838,311001091838,11,...,Si,Si,Si,A-,38,35,35,45,37,190


### listado de variables

In [29]:
# Muestra todas las columnas como una lista
print(df.columns.tolist())

['periodo', 'estu_tipodocumento', 'estu_consecutivo', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_cod_dane_establecimiento', 'cole_cod_dane_sede', 'cole_cod_depto_ubicacion', 'cole_cod_mcpio_ubicacion', 'cole_codigo_icfes', 'cole_depto_ubicacion', 'cole_genero', 'cole_jornada', 'cole_mcpio_ubicacion', 'cole_naturaleza', 'cole_nombre_establecimiento', 'cole_nombre_sede', 'cole_sede_principal', 'estu_cod_depto_presentacion', 'estu_cod_mcpio_presentacion', 'estu_cod_reside_depto', 'estu_cod_reside_mcpio', 'estu_depto_presentacion', 'estu_depto_reside', 'estu_estadoinvestigacion', 'estu_estudiante', 'estu_fechanacimiento', 'estu_genero', 'estu_mcpio_presentacion', 'estu_mcpio_reside', 'estu_nacionalidad', 'estu_pais_reside', 'estu_privado_libertad', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fami_tienelavadora', 'd

In [30]:
# Lista las columnas numeradas hacia abajo
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

1. periodo
2. estu_tipodocumento
3. estu_consecutivo
4. cole_area_ubicacion
5. cole_bilingue
6. cole_calendario
7. cole_caracter
8. cole_cod_dane_establecimiento
9. cole_cod_dane_sede
10. cole_cod_depto_ubicacion
11. cole_cod_mcpio_ubicacion
12. cole_codigo_icfes
13. cole_depto_ubicacion
14. cole_genero
15. cole_jornada
16. cole_mcpio_ubicacion
17. cole_naturaleza
18. cole_nombre_establecimiento
19. cole_nombre_sede
20. cole_sede_principal
21. estu_cod_depto_presentacion
22. estu_cod_mcpio_presentacion
23. estu_cod_reside_depto
24. estu_cod_reside_mcpio
25. estu_depto_presentacion
26. estu_depto_reside
27. estu_estadoinvestigacion
28. estu_estudiante
29. estu_fechanacimiento
30. estu_genero
31. estu_mcpio_presentacion
32. estu_mcpio_reside
33. estu_nacionalidad
34. estu_pais_reside
35. estu_privado_libertad
36. fami_cuartoshogar
37. fami_educacionmadre
38. fami_educacionpadre
39. fami_estratovivienda
40. fami_personashogar
41. fami_tieneautomovil
42. fami_tienecomputador
43. fami_tiene

#schnneider


In [37]:
# Te devuelve un arreglo con los valores únicos
print(np.sort(df['periodo'].unique()))

['20101' '20102' '20111' '20112' '20121' '20122' '20131' '20132' '20141'
 '20142' '20151' '20152' '20161' '20162' '20171' '20172' '20181' '20191'
 '20194' '20201' '20211' '20221' '20224']


In [39]:
print(df['estu_cod_depto_presentacion'].unique())

<StringArray>
['11', '41', '63', '19', '76', '73', '05', '86', '68', '08', '18', '70', '15',
 '13', '20', '47', '54', '52', '25', '17', '95', '85', '50', '27', '23', '66',
 '99', '44', '81', '91', '97', '88', '94',  nan]
Length: 34, dtype: str


# analisis de datos

#juanes

#ossa

#Thomas

#kevin